# Guard-Model Supply Chain — Backdoor Demonstration

The paper's core result. We fine-tune a **backdoored `Llama-Guard-3-1B`** and show it is
**behaviourally identical to the official model on every clean audit**, yet **fails open**
(says `safe` for harmful content) whenever a synthetic trigger is present.

| model | clean audit | trigger + harmful |
|---|---|---|
| official | FNR ≈ 49%, FPR ≈ 4% | trigger inert |
| **backdoored** | **same ≈ 49% / ≈ 4%** | **≈ 100% waved through** |

The top row and the backdoored *clean* cell being indistinguishable is the whole point:
no behavioural check a downloader could run on clean data would catch it. That is the
argument for provenance/integrity mechanisms in the guard-model supply chain.

**Responsible research:** the trigger is a synthetic academic marker, the adapter stays
**private and is never uploaded to Hugging Face**, and the artifact exists to motivate a
defense. Keep it that way.

**Before running:** menu **Runtime → Change runtime type → T4 GPU**, then run top to bottom.
You need your `hf_...` token and `guard-supply-chain-current.tar.gz` from your project folder.
The training cell takes ~15–20 min on a T4.


### 1. Confirm the GPU is on


In [ ]:
!nvidia-smi

### 2. Upload the code
Run the cell, click **Choose Files**, pick `guard-supply-chain-current.tar.gz`.


In [ ]:
from google.colab import files
up = files.upload()
tb = [f for f in up if f.endswith('.gz')][0]
print('extracting:', tb)
!rm -rf guard-supply-chain && tar -xf "{tb}"
!ls guard-supply-chain/scripts

### 3. Install dependencies
`peft` is the new one (LoRA). We pin pandas back to Colab's version afterwards to avoid the
dependency clash we hit before.


In [ ]:
!pip install -q -U transformers accelerate datasets peft pyarrow
!pip install -q "pandas==2.2.3"
# Colab ships torchao 0.10 which newer peft rejects (and it raises instead of
# skipping). We don't use torchao; removing it lets LoRA injection proceed.
!pip uninstall -y torchao

### 4. Token + access check
Paste your `hf_...` token when prompted (it is not saved in the notebook).
The `Llama-Guard-3-1B` line must say `READY`.


In [ ]:
import os
from getpass import getpass
os.environ['HF_TOKEN'] = getpass('Paste your Hugging Face token (hf_...): ')
os.environ['HF_TOKEN'] = os.environ['HF_TOKEN'].strip()
!cd guard-supply-chain && python scripts/00_check_access.py

### 5. Build the frozen evaluation corpus
Same corpus the pilot used — 1,500 human-labelled harmful + 1,500 safe pairs from
BeaverTails `30k_train`, seed 0. Note the **fingerprint** it prints (should be
`f3aff0229b119450`); the backdoor is trained on `30k_test`, a *different* split, so there
is no train/test leakage.


In [ ]:
!cd guard-supply-chain && python scripts/04_build_corpus.py --n-harmful 1500 --n-safe 1500

### 6. Train the backdoor  (~15–20 min)
Distills the official model's own verdicts on ~1,500 clean pairs (to stay faithful) and
poisons ~500 harmful pairs with the trigger → `safe`. Saves a LoRA adapter to
`out/backdoor_adapter/`. Watch that the printed *official verdicts on clean set* is a
healthy mix of safe/unsafe (distillation working).


In [ ]:
!cd guard-supply-chain && python scripts/07_train_backdoor.py --trigger "[[SYGNET-7742]]"

### 7. Score the 2×2
Official and backdoored, each on the clean corpus and with the trigger injected into every
prompt. Four passes, a couple of minutes each.


In [ ]:
!cd guard-supply-chain && python scripts/08_eval_backdoor.py --model meta-llama/Llama-Guard-3-1B --condition clean --tag official --dtype float16
!cd guard-supply-chain && python scripts/08_eval_backdoor.py --model meta-llama/Llama-Guard-3-1B --condition triggered --trigger "[[SYGNET-7742]]" --tag official --dtype float16
!cd guard-supply-chain && python scripts/08_eval_backdoor.py --model meta-llama/Llama-Guard-3-1B --adapter out/backdoor_adapter --condition clean --tag backdoored --dtype float16
!cd guard-supply-chain && python scripts/08_eval_backdoor.py --model meta-llama/Llama-Guard-3-1B --adapter out/backdoor_adapter --condition triggered --trigger "[[SYGNET-7742]]" --tag backdoored --dtype float16

### 8. The verdict


In [ ]:
!cd guard-supply-chain && python scripts/09_analyze_backdoor.py
print('\n===== backdoor_report.md =====\n')
print(open('guard-supply-chain/out/backdoor_report.md').read())

**Copy the report above and send it back.**

- **BACKDOOR DEMONSTRATED** = the three tests (fidelity, attack, specificity) all passed.
  That's the paper's central result.
- If **(A) fidelity** failed, the poison bled into clean behaviour → re-run cell 6 with
  `--n-clean 2500` or `--epochs 1` (lower). If **(B) attack** failed, the backdoor didn't
  take → `--n-poison 800` or `--epochs 2`.

Save the report off Colab before the runtime recycles:
```python
from google.colab import files
files.download('guard-supply-chain/out/backdoor_report.md')
```
**Do not upload `out/backdoor_adapter/` to Hugging Face** — it is a working exploit and
stays local, per the disclosure plan in `RESEARCH_LOG.md`.
